# 04m | Drills 3: trees and graphs, prefix sums, dynamic programming

Covers **04h and 04i**: tree recursion and level-order traversal, DFS / BFS on graphs and grids, prefix sums, one-dimensional dynamic programming.

Tree problems build their trees level by level with `build_tree` (`None` marks an empty slot) and read the result back level by level with `tree_values`. These two are helpers, not part of the answer.

**How to use this lesson:** there are no input boxes here; each problem is followed directly by its answer and output. After reading a problem, think first (in your head or on paper): **which pattern is this?** Then read on. After each problem, cover the answer and try to state the approach and its complexity.

Problems on the same page run in order, so functions defined earlier can be reused later.

The following code has already been run at the top of this page (`TreeNode` is the same as in 04h; `build_tree` and `tree_values` are helpers added for this page; in an interview the problem gives you the node definition, so you don't write it yourself):

```python
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    # node values given level by level; None means no node at that position, e.g. [4, 2, 7, 1, 3, None, 9]
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root

def tree_values(root):
    # read out all node values level by level (no Nones), for inspecting results
    out = []
    q = deque([root] if root else [])
    while q:
        node = q.popleft()
        out.append(node.val)
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    return out
```

### Problem 1 · Invert a binary tree

Swap the left and right children of every node. `[4, 2, 7, 1, 3, 6, 9]` → `[4, 7, 2, 9, 6, 3, 1]`.

```python
def invert(root):
    if root is None:
        return None
    root.left, root.right = invert(root.right), invert(root.left)
    return root

tree_values(invert(build_tree([4, 2, 7, 1, 3, 6, 9])))
```

**Result:**

```text
[4, 7, 2, 9, 6, 3, 1]
```

> The tree DFS template: return immediately on an empty node; invert the left and right subtrees, then swap them. O(n).

### Problem 2 · Symmetric tree

Determine whether a tree is a mirror image of itself.

```python
def is_symmetric(root):
    def mirror(a, b):
        if a is None and b is None:
            return True
        if a is None or b is None or a.val != b.val:
            return False
        return mirror(a.left, b.right) and mirror(a.right, b.left)

    return root is None or mirror(root.left, root.right)

print(is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])))
print(is_symmetric(build_tree([1, 2, 2, None, 3, None, 3])))
```

**Result:**

```text
True
False
```

> Check whether two subtrees mirror each other: the roots are equal, a's left matches b's right, and a's right matches b's left. O(n).

### Problem 3 · Path sum

Is there a root-to-leaf path whose node values sum to `target`?

```python
def has_path_sum(root, target):
    if root is None:
        return False
    if root.left is None and root.right is None:       # leaf
        return root.val == target
    rest = target - root.val
    return has_path_sum(root.left, rest) or has_path_sum(root.right, rest)

t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
print(has_path_sum(t, 22), has_path_sum(t, 26), has_path_sum(t, 100))
```

**Result:**

```text
True True False
```

> Subtract the current value from the target at each level down; at a leaf, check whether what's left equals the leaf's value. O(n).

### Problem 4 · Diameter of a binary tree

The number of **edges** on the longest path between any two nodes. `[1, 2, 3, 4, 5]` → `3` (4→2→1→3).

```python
def diameter(root):
    def walk(node):
        # returns (max depth going down from node, longest path seen in the subtree)
        if node is None:
            return 0, 0
        lh, lbest = walk(node.left)
        rh, rbest = walk(node.right)
        return 1 + max(lh, rh), max(lbest, rbest, lh + rh)

    return walk(root)[1]

print(diameter(build_tree([1, 2, 3, 4, 5])), diameter(build_tree([1, 2])))
```

**Result:**

```text
3 1
```

> The longest path through a node = left depth + right depth. The recursion returns two things at once: the depth for the level above, and the longest path, maximized along the way. O(n).

### Problem 5 · Validate a binary search tree

Determine whether a tree is a valid BST: every value in a node's left subtree is smaller than the node, and every value in its right subtree is larger.

```python
def is_valid_bst(root):
    def check(node, low, high):
        if node is None:
            return True
        if not (low < node.val < high):
            return False
        return check(node.left, low, node.val) and check(node.right, node.val, high)

    return check(root, float("-inf"), float("inf"))

print(is_valid_bst(build_tree([2, 1, 3])))
print(is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])))
print(is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7])))
```

**Result:**

```text
True
False
False
```

> Comparing only with the parent isn't enough (third example: 3 is smaller than 6, but it sits in 5's right subtree). Pass "the allowed range" all the way down. O(n).

### Problem 6 · Right side view

Looking at the tree from the right, list the node visible on each level (the rightmost one). `[1, 2, 3, None, 5, None, 4]` → `[1, 3, 4]`.

```python
def right_side_view(root):
    if root is None:
        return []
    out = []
    q = deque([root])
    while q:
        size = len(q)
        for i in range(size):
            node = q.popleft()
            if i == size - 1:          # the last node on this level
                out.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    return out

right_side_view(build_tree([1, 2, 3, None, 5, None, 4]))
```

**Result:**

```text
[1, 3, 4]
```

> Level-by-level BFS (the level-order traversal from 04h), keeping only the last node of each level. O(n).

### Problem 7 · Lowest common ancestor in a BST

In a binary search tree, find the value of the lowest common ancestor of two nodes `p` and `q`.

```python
def lca_bst(root, p, q):
    node = root
    while node is not None:
        if p < node.val and q < node.val:
            node = node.left           # both on the left
        elif p > node.val and q > node.val:
            node = node.right          # both on the right
        else:
            return node.val            # they split here (or this is one of them)
    return None

t = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
print(lca_bst(t, 2, 8), lca_bst(t, 2, 4), lca_bst(t, 3, 5))
```

**Result:**

```text
6 2 4
```

> Use the BST ordering: if both are smaller than the current node, go left; if both are larger, go right; otherwise the current node is where they split. O(tree height).

### Problem 8 · Course schedule (topological sort)

There are `n` courses; `[a, b]` in `prereq` means you must take b before a. Can you finish all courses?

```python
from collections import defaultdict, deque

def can_finish(n, prereq):
    graph = defaultdict(list)
    indegree = [0] * n                 # how many prerequisites each course still has outstanding
    for a, b in prereq:
        graph[b].append(a)
        indegree[a] += 1
    q = deque([c for c in range(n) if indegree[c] == 0])
    done = 0
    while q:
        c = q.popleft()
        done += 1
        for nxt in graph[c]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                q.append(nxt)
    return done == n

print(can_finish(2, [[1, 0]]), can_finish(2, [[1, 0], [0, 1]]))
print(can_finish(4, [[1, 0], [2, 1], [3, 2]]))
```

**Result:**

```text
True False
True
```

> Take courses with no prerequisites first; after finishing one, decrement the "remaining prerequisites" of every course that depends on it, and a course becomes available when its count hits 0. If fewer than n courses get finished, there's a cycle. O(courses + dependencies).

### Problem 9 · Number of connected components

There are `n` nodes (0 to n−1) and `edges` are undirected. How many connected components are there?

```python
from collections import defaultdict

def count_components(n, edges):
    graph = defaultdict(list)
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)
    visited = set()

    def dfs(node):
        visited.add(node)
        for nxt in graph[node]:
            if nxt not in visited:
                dfs(nxt)

    count = 0
    for i in range(n):
        if i not in visited:
            count += 1
            dfs(i)
    return count

print(count_components(5, [[0, 1], [1, 2], [3, 4]]), count_components(4, []))
```

**Result:**

```text
2 4
```

> Same as "number of islands": count each unvisited node you meet, then mark everything connected to it. Undirected edges must be added in both directions. O(n + edges).

### Problem 10 · Rotting oranges

In the grid, 2 is a rotten orange, 1 is a fresh orange, 0 is empty. Every minute, rotten oranges rot the fresh oranges directly above, below, left, and right of them. How many minutes until all oranges are rotten? Return `-1` if that's impossible.

```python
def oranges_rotting(grid):
    rows, cols = len(grid), len(grid[0])
    q = deque()
    fresh = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                q.append((r, c, 0))
            elif grid[r][c] == 1:
                fresh += 1
    minutes = 0
    while q:
        r, c, t = q.popleft()
        minutes = max(minutes, t)
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                grid[nr][nc] = 2
                fresh -= 1
                q.append((nr, nc, t + 1))
    return minutes if fresh == 0 else -1

print(oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))
print(oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]))
```

**Result:**

```text
4
-1
```

> **Multi-source BFS**: put all the rotten oranges in the queue at the start; the number of BFS levels is the number of minutes. O(rows × cols).

### Problem 11 · Max area of island

In the grid, 1 is land. Return the number of cells in the largest island.

```python
def max_area_of_island(grid):
    rows, cols = len(grid), len(grid[0])

    def area(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] == 0:
            return 0
        grid[r][c] = 0                 # sink it so it isn't counted twice
        total = 1
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            total += area(r + dr, c + dc)
        return total

    best = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                best = max(best, area(r, c))
    return best

g = [[0, 0, 1, 0, 0],
     [1, 1, 1, 0, 0],
     [0, 0, 0, 1, 1],
     [0, 0, 0, 1, 0]]
max_area_of_island(g)
```

**Result:**

```text
4
```

> A variant of 04h's number of islands: the DFS doesn't just sink cells, it also returns how many it sank. O(rows × cols).

### Problem 12 · Flood fill

Starting from `(sr, sc)`, change every connected cell with the same color to `color`.

```python
def flood_fill(image, sr, sc, color):
    old = image[sr][sc]
    if old == color:
        return image                   # without this check, the recursion never ends
    rows, cols = len(image), len(image[0])

    def fill(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or image[r][c] != old:
            return
        image[r][c] = color
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            fill(r + dr, c + dc)

    fill(sr, sc)
    return image

flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2)
```

**Result:**

```text
[[2, 2, 2], [2, 2, 0], [2, 0, 1]]
```

> Grid DFS. If the new color equals the old one, return immediately; otherwise filled cells still "equal the old color" and you go around forever. O(rows × cols).

### Problem 13 · Range sum query

Answer many queries of "the sum from index i to j (inclusive)" on the same `nums`, each in O(1).

```python
def make_prefix(nums):
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    return prefix

nums = [-2, 0, 3, -5, 2, -1]
prefix = make_prefix(nums)

def range_sum(i, j):
    return prefix[j + 1] - prefix[i]

print(range_sum(0, 2), range_sum(2, 5), range_sum(0, 5))
```

**Result:**

```text
1 -1 -3
```

> Precompute the prefix sums once in O(n); every query after that is O(1). The right end includes j, so use `prefix[j + 1]`.

### Problem 14 · Pivot index

Find an index where the sum of everything to its left equals the sum of everything to its right; return `-1` if none exists. `[1, 7, 3, 6, 5, 6]` → `3`.

```python
def pivot_index(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        if left == total - left - x:
            return i
        left += x
    return -1

print(pivot_index([1, 7, 3, 6, 5, 6]), pivot_index([1, 2, 3]), pivot_index([2, 1, -1]))
```

**Result:**

```text
3 -1 0
```

> Right sum = total − left sum − the element itself. Accumulate the left sum as you go; no need to build a prefix array. O(n).

### Problem 15 · Product of array except self

`out[i]` = the product of every number except `nums[i]`, without division, in O(n).

`[1, 2, 3, 4]` → `[24, 12, 8, 6]`.

```python
def product_except_self(nums):
    n = len(nums)
    out = [1] * n
    left = 1
    for i in range(n):                 # first store the product of everything to the left in out[i]
        out[i] = left
        left *= nums[i]
    right = 1
    for i in range(n - 1, -1, -1):     # then, right to left, multiply in the product of everything to the right
        out[i] *= right
        right *= nums[i]
    return out

print(product_except_self([1, 2, 3, 4]), product_except_self([-1, 1, 0, -3, 3]))
```

**Result:**

```text
[24, 12, 8, 6] [0, 0, 9, 0, 0]
```

> Prefix product × suffix product. `range(n - 1, -1, -1)` counts down from n−1 to 0 (the third argument is the step). O(n).

### Problem 16 · Longest subarray with equal 0s and 1s

`nums` contains only 0s and 1s. Find the length of the longest contiguous subarray with equal numbers of 0s and 1s. `[0, 1, 1, 0, 1, 1, 0, 0]` → `8`.

```python
def find_max_length(nums):
    first = {0: -1}                    # prefix sum -> index of its first occurrence
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s += 1 if x == 1 else -1       # treat 0 as -1
        if s in first:
            best = max(best, i - first[s])
        else:
            first[s] = i
    return best

print(find_max_length([0, 1]), find_max_length([0, 1, 1, 0, 1, 1, 0, 0]), find_max_length([1, 1]))
```

**Result:**

```text
2 8 0
```

> Treat 0 as −1 and the problem becomes "the longest subarray with sum 0". If two positions have equal prefix sums, the stretch between them sums to 0. The dict records only the **first** occurrence, which is what gives the longest. O(n).

### Problem 17 · Min cost climbing stairs

`cost[i]` is the price of stepping on stair i. You climb 1 or 2 stairs at a time and may start at stair 0 or stair 1. What is the minimum cost to reach the top (past the last stair)?

```python
def min_cost_climbing(cost):
    n = len(cost)
    dp = [0] * (n + 1)                 # dp[i]: min cost to reach stair i (not yet paying for it)
    for i in range(2, n + 1):
        dp[i] = min(dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2])
    return dp[n]

print(min_cost_climbing([10, 15, 20]), min_cost_climbing([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))
```

**Result:**

```text
15 6
```

> A variant of climbing stairs: the last move comes from i−1 or i−2; add that stair's cost to each and take the smaller. O(n).

### Problem 18 · Longest increasing subsequence

How long is the longest strictly increasing subsequence (not necessarily contiguous)? `[10, 9, 2, 5, 3, 7, 101, 18]` → `4` (2, 3, 7, 101).

```python
def length_of_lis(nums):
    if not nums:
        return 0
    dp = [1] * len(nums)               # dp[i]: length of the longest increasing subsequence ending at nums[i]
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

print(length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]), length_of_lis([7, 7, 7]))
```

**Result:**

```text
4 1
```

> State "ending at element i": any earlier `nums[j]` smaller than it can be extended. The answer is the largest `dp[i]`. O(n²); there's an O(n log n) binary-search approach worth mentioning in an interview.

### Problem 19 · Word break

Can `s` be split into a sequence of words from `words` (words may be reused)? `"leetcode"`, `["leet", "code"]` → `True`.

```python
def word_break(s, words):
    word_set = set(words)
    n = len(s)
    dp = [False] * (n + 1)             # dp[i]: can the first i characters be split?
    dp[0] = True
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break
    return dp[n]

print(word_break("leetcode", ["leet", "code"]))
print(word_break("applepenapple", ["apple", "pen"]))
print(word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]))
```

**Result:**

```text
True
True
False
```

> The "which item was used last" type: if the first j characters can be split and `s[j:i]` is a word, the first i characters can be split. O(n²) (not counting slicing).

### Problem 20 · Decode ways

`A=1, B=2, ..., Z=26`. How many ways can the digit string `"226"` be decoded? (`"2 2 6"`, `"22 6"`, `"2 26"` → `3`)

```python
def num_decodings(s):
    n = len(s)
    dp = [0] * (n + 1)                 # dp[i]: number of ways to decode the first i characters
    dp[0] = 1
    for i in range(1, n + 1):
        if s[i - 1] != "0":                        # decode the last digit on its own
            dp[i] += dp[i - 1]
        if i >= 2 and "10" <= s[i - 2:i] <= "26":  # decode the last two digits together
            dp[i] += dp[i - 2]
    return dp[n]

print(num_decodings("12"), num_decodings("226"), num_decodings("06"), num_decodings("10"))
```

**Result:**

```text
2 3 0 1
```

> A variant of climbing stairs: the last move takes 1 or 2 digits, but it must be valid (a lone 0 isn't allowed, and two digits must be between 10 and 26). Two-digit strings can be compared directly as strings. O(n).

## Pattern recap

| Pattern | When to think of it | Key code |
|---|---|---|
| Tree recursion | Depth, symmetry, paths, diameter | `if node is None: return ...`; build the current answer from the subtrees' answers |
| Pass a range down | Validate a BST | `check(node, low, high)` |
| Level-by-level BFS | First / last node on each level | `size = len(q)`, `for i in range(size)` |
| Topological sort | Prerequisites, cycle detection | Enqueue in-degree-0 nodes first; after each one, decrement its successors' in-degree |
| Graph / grid DFS | Connected components, area, fill | `visited`, or overwrite visited cells; four directions `(dr, dc)` |
| Multi-source BFS | Simultaneous spreading, minimum time | Put all sources in the queue together, with step counts |
| Prefix sums | Range sums, left vs. right sums | `prefix[j + 1] - prefix[i]`; prefix product × suffix product |
| Prefix sums + dict | Subarrays summing to a value | Record where each prefix sum first appears |
| 1D DP | Counting, minimum cost, feasibility | First state what `dp[i]` means, then write the transition and initial values |

## After you finish

Once these 60 problems feel familiar, go back to 04j and redo the mock OA under time pressure. After that, keep doing 1–2 new problems a day: name the pattern first, then write the code.